In [ ]:
import numpy as np

from statsmodels.gam.smooth_basis import (
    GenericSmoothers,
    UnivariateBSplines,
    UnivariateCubicCyclicSplines
)

# Use the same rows and ordering as the model dataframe
x = df[
    ["UV", "RH", "WS", "temp", "SR", "Year", "Week"]
].to_numpy()

# Five weather-variable B-splines
uv = UnivariateBSplines(
    x[:, 0], df=8, degree=3,
    include_intercept=False,
    variable_name="UV"
)

rh = UnivariateBSplines(
    x[:, 1], df=20, degree=3,
    include_intercept=False,
    variable_name="RH"
)

ws = UnivariateBSplines(
    x[:, 2], df=20, degree=3,
    include_intercept=False,
    variable_name="WS"
)

temp = UnivariateBSplines(
    x[:, 3], df=20, degree=3,
    include_intercept=False,
    variable_name="temp"
)

sr = UnivariateBSplines(
    x[:, 4], df=20, degree=3,
    include_intercept=False,
    variable_name="SR"
)

# Two cyclic splines
year = UnivariateCubicCyclicSplines(
    x[:, 5],
    df=your_original_year_df,
    constraints="center",
    variable_name="Year"
)

week = UnivariateCubicCyclicSplines(
    x[:, 6],
    df=your_original_week_df,
    constraints="center",
    variable_name="Week"
)

# Combine all seven smoothers
smoother = GenericSmoothers(
    x,
    [uv, rh, ws, temp, sr, year, week]
)

In [ ]:
gam = GLMGam.from_formula(
    formula="y ~ 1",
    data=df,
    smoother=smoother,
    alpha=[
        alpha_uv,
        alpha_rh,
        alpha_ws,
        alpha_temp,
        alpha_sr,
        100.0,
        50.0
    ],
    family=sm.families.NegativeBinomial()
)

---


In [ ]:
import scipy.stats as stats
mm = GLMGam.from_formula(f"H1 ~ {' + '.join(exp_name_list)} + pandemic_flag + C(Year)", data=model_data, smoother=smoothers, alpha=[5, 5, 5, 5], family=sm.families.NegativeBinomial(alpha=0.5)).fit()
mu = mm.predict()

y = df["H1"].values
cdf_lower = stats.poisson.cdf(y-1, mu)
cdf_higher = stats.poisson.cdf(y, mu)
uniform_residual = np.random.uniform(cdf_lower, cdf_higher)
normal_quantile_residuals = stats.norm.ppf(uniform_residual)
sm.qqplot(normal_quantile_residuals, line='45')
plt.show()

pearson_chi2 = mm.pearson_chi2
df_residual = mm.df_resid
dispersion_ratio = pearson_chi2 / df_residual
print(f"Ratio: {dispersion_ratio}")
observed_0 = np.mean(y == 0)
predicted_0 = np.mean(stats.poisson.pmf(0, mu))

print(f"Observed 0: {observed_0}")

print(f"Predicted 0: {predicted_0}")

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import statsmodels.api as sm

# 1. Get predicted means (mu) and the estimated alpha (overdispersion parameter)
mu = mm.predict()
y = df['H1'].values # replace 'y' with your response variable name

# Extract alpha from the fitted GLM/GLMGam model result
# Note: For sm.families.NegativeBinomial, alpha is family.alpha or tmp_result.scale
alpha = tmp_model.family.alpha 

# 2. Convert mu and alpha to scipy's Negative Binomial parameters (n, p)
# Var = mu + alpha * mu^2
# In scipy: n = 1/alpha, p = 1 / (1 + alpha * mu)
n = 1.0 / alpha
p = 1.0 / (1.0 + alpha * mu)

# 3. Calculate Randomized Quantile Residuals using the correct NB CDF
cdf_lower = stats.nbinom.cdf(y - 1, n, p)
cdf_upper = stats.nbinom.cdf(y, n, p)

# Randomize to make it continuous
uniform_residuals = np.random.uniform(cdf_lower, cdf_upper)

# Convert to normal-quantiles for the Q-Q plot
normal_quantile_residuals = stats.norm.ppf(uniform_residuals)

# 4. Plot the corrected Q-Q Plot
sm.qqplot(normal_quantile_residuals, line='45')
plt.title("Corrected Quantile Residual Q-Q Plot (Negative Binomial)")
plt.show()